# 📝 Notebook 03 — Abstractive Summarization

LoRA fine-tune and evaluate three architectures on the **amharic-eval-500** set:
- **mBERT** → BERT2BERT EncoderDecoderModel (cross-attn randomly init.)
- **mT5-small** → native seq2seq
- **BLOOMZ-560M** → causal LM with prompt-target concatenation

Metrics: ROUGE-1/2/L (Amharic-aware), sacreBLEU, BERTScore (xlm-roberta-large).

In [ ]:
import importlib, subprocess, sys
for pkg in ['torch','transformers','peft','datasets','accelerate','sentencepiece',
            'sacrebleu','bert-score','codecarbon','pyyaml','nvidia-ml-py']:
    if not importlib.util.find_spec(pkg.replace('-','_')):
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, torch

REPO = Path('.').resolve()
if REPO.name == 'notebooks': REPO = REPO.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)

from src.utils import load_config, set_all_seeds, MODEL_KEYS, ARCH_LABEL
cfg = load_config()
SEED = cfg['experiment']['seeds'][0]
set_all_seeds(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
data_dir = Path(cfg['experiment']['processed_data_dir'])
if not (data_dir / 'manifest.json').exists():
    from src.preprocessing.data_prep import prepare_all
    prepare_all(cfg, SEED)
eval_df = pd.read_csv(data_dir / 'summ_eval500.csv')
train_df = pd.read_csv(data_dir / 'summ_train.csv')
print(f'Train: {len(train_df)}   Eval: {len(eval_df)}')

## 1. Train & Evaluate Each Model

In [ ]:
from src.training.summarization import run_summarization

results = {}
for model_key in MODEL_KEYS:
    print(f'\n{"="*60}\n  {ARCH_LABEL[model_key]} — {cfg["models"][model_key]["hf_id"]}\n{"="*60}')
    try:
        out_dir = run_summarization(cfg, model_key, SEED)
        results[model_key] = {
            'metrics': json.loads((out_dir / 'metrics.json').read_text()),
            'predictions': pd.read_csv(out_dir / 'predictions.csv'),
        }
    except Exception as e:
        print(f'  FAILED: {e}')
        results[model_key] = None

## 2. Results

In [ ]:
rows = []
for m in MODEL_KEYS:
    r = results.get(m)
    if r is None: continue
    s = r['metrics']
    rows.append({'Architecture': ARCH_LABEL[m],
                 'ROUGE-1': f"{s['rouge1']:.4f}", 'ROUGE-2': f"{s['rouge2']:.4f}",
                 'ROUGE-L': f"{s['rougeL']:.4f}", 'sacreBLEU': f"{s['sacrebleu']:.2f}",
                 'BERTScore-F1': f"{s.get('bertscore_f1','n/a')}",
                 'Empty pred %': f"{s['empty_pred_rate']*100:.1f}"})
print(pd.DataFrame(rows).to_markdown(index=False))

### 2.1 Sample Predictions

In [ ]:
avail = [m for m in MODEL_KEYS if results.get(m)]
for i in range(min(5, len(eval_df))):
    print(f'\n--- Example {i} ---')
    print(f'Source (first 200 chars): {eval_df.iloc[i]["source"][:200]}…')
    print(f'Reference: {eval_df.iloc[i]["target"]}')
    for m in avail:
        print(f'  {ARCH_LABEL[m]}: {results[m]["predictions"].iloc[i]["prediction"]}')

### 2.2 Metric Comparison (Radar Chart)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

metrics_keys = ['rouge1','rouge2','rougeL']
if any(results.get(m) and 'bertscore_f1' in results[m]['metrics'] for m in avail):
    metrics_keys.append('bertscore_f1')

angles = np.linspace(0, 2*np.pi, len(metrics_keys), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(subplot_kw={'polar': True}, figsize=(5, 5))
colors = {'mbert':'#1f77b4','mt5':'#2ca02c','bloomz':'#d62728'}
for m in avail:
    vals = [results[m]['metrics'].get(k, 0) for k in metrics_keys] + [results[m]['metrics'].get(metrics_keys[0], 0)]
    ax.plot(angles, vals, 'o-', label=ARCH_LABEL[m], color=colors[m])
    ax.fill(angles, vals, alpha=0.1, color=colors[m])
ax.set_thetagrids([a*180/np.pi for a in angles[:-1]], metrics_keys)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('Summarization Metrics')
plt.tight_layout(); plt.show()

## 3. Discussion

**Key observations:**
- **mT5** is expected to perform best: it was pre-trained on span corruption across 101 languages (including Amharic) and is natively seq2seq.
- **BLOOMZ** can generate Amharic text but requires careful prompting; its causal architecture may produce less abstractive summaries.
- **mBERT (BERT2BERT)** has randomly initialised cross-attention, placing it at a structural disadvantage. Its scores reflect the cost of absent generative pre-training.

See `docs/limitations.md` for caveats about the evaluation methodology (single reference, unstemmed ROUGE, small eval set).